# Neural Network from Scratch with NumPy

This notebook implements a small feedforward neural network for the XOR classification problem.

The implementation uses only NumPy and demonstrates the main stages of neural-network learning:

1. initialize the weights;
2. perform a forward pass;
3. calculate prediction error;
4. propagate the error backward;
5. update the weights;
6. repeat the process until the network learns the XOR mapping.

## Learning Objectives

After completing this notebook, students should be able to:

- explain the structure of a fully connected neural network;
- apply the sigmoid activation function;
- perform forward propagation using matrix multiplication;
- compute gradients with backpropagation;
- update network weights with gradient descent;
- evaluate binary predictions;
- explain why XOR requires a hidden layer.

## Network Architecture

The network contains:

- **3 input values**: two XOR features and one constant bias feature;
- **4 hidden neurons**;
- **1 output neuron**;
- sigmoid activation in the hidden and output layers.

The third input column is fixed at `1` and acts as an input-level bias term.

## 1. Import NumPy and define the XOR dataset

In [ ]:
import numpy as np


X = np.array(
    [
        [0, 0, 1],
        [0, 1, 1],
        [1, 0, 1],
        [1, 1, 1],
    ],
    dtype=float,
)

y = np.array(
    [
        [0],
        [1],
        [1],
        [0],
    ],
    dtype=float,
)

print("Input shape:", X.shape)
print("Target shape:", y.shape)

## 2. Define the sigmoid activation

The sigmoid function maps any real-valued input to the interval `(0, 1)`:

\[
\sigma(z) = \frac{1}{1 + e^{-z}}
\]

When the sigmoid output has already been calculated, its derivative is:

\[
\sigma'(z) = \sigma(z)(1 - \sigma(z))
\]

The derivative is used during backpropagation to determine how strongly each weight contributed to the prediction error.

In [ ]:
def sigmoid(values):
    return 1.0 / (1.0 + np.exp(-values))


def sigmoid_derivative(activated_values):
    return activated_values * (1.0 - activated_values)

## 3. Implement the neural network

In [ ]:
class NeuralNetwork:
    def __init__(
        self,
        input_size,
        hidden_size,
        output_size,
        *,
        learning_rate=0.5,
        random_seed=42,
    ):
        self.learning_rate = learning_rate

        random_generator = np.random.default_rng(
            random_seed
        )

        self.weights_input_hidden = (
            random_generator.uniform(
                -1.0,
                1.0,
                size=(input_size, hidden_size),
            )
        )

        self.weights_hidden_output = (
            random_generator.uniform(
                -1.0,
                1.0,
                size=(hidden_size, output_size),
            )
        )

    def forward(self, inputs):
        hidden_linear = (
            inputs @ self.weights_input_hidden
        )
        hidden_output = sigmoid(hidden_linear)

        output_linear = (
            hidden_output
            @ self.weights_hidden_output
        )
        predictions = sigmoid(output_linear)

        return hidden_output, predictions

    def train(self, inputs, targets, epochs=10_000):
        loss_history = []
        sample_count = len(inputs)

        for epoch in range(epochs):
            hidden_output, predictions = (
                self.forward(inputs)
            )

            output_error = targets - predictions

            output_delta = (
                output_error
                * sigmoid_derivative(predictions)
            )

            hidden_error = (
                output_delta
                @ self.weights_hidden_output.T
            )

            hidden_delta = (
                hidden_error
                * sigmoid_derivative(hidden_output)
            )

            self.weights_hidden_output += (
                self.learning_rate
                * hidden_output.T
                @ output_delta
                / sample_count
            )

            self.weights_input_hidden += (
                self.learning_rate
                * inputs.T
                @ hidden_delta
                / sample_count
            )

            mean_squared_error = np.mean(
                output_error ** 2
            )
            loss_history.append(
                float(mean_squared_error)
            )

        return loss_history

    def predict_probabilities(self, inputs):
        _, predictions = self.forward(inputs)
        return predictions

    def predict(self, inputs, threshold=0.5):
        probabilities = self.predict_probabilities(
            inputs
        )
        return (
            probabilities >= threshold
        ).astype(int)

## 4. Train the network

During each epoch, the network processes all four XOR examples, computes the prediction error, and updates both weight matrices.

A fixed random seed makes the demonstration reproducible.

In [ ]:
network = NeuralNetwork(
    input_size=X.shape[1],
    hidden_size=4,
    output_size=y.shape[1],
    learning_rate=0.5,
    random_seed=42,
)

loss_history = network.train(
    X,
    y,
    epochs=10_000,
)

print("Initial loss:", round(loss_history[0], 6))
print("Final loss:", round(loss_history[-1], 6))

## 5. Evaluate the learned XOR mapping

In [ ]:
probabilities = network.predict_probabilities(X)
predictions = network.predict(X)

accuracy = np.mean(predictions == y)

for features, probability, prediction, target in zip(
    X[:, :2],
    probabilities.ravel(),
    predictions.ravel(),
    y.ravel(),
):
    print(
        f"Input={features.astype(int).tolist()} "
        f"Probability={probability:.4f} "
        f"Prediction={prediction} "
        f"Target={int(target)}"
    )

print(f"Accuracy: {accuracy:.2%}")

## Result Interpretation

The network should learn the following XOR relationship:

| Input 1 | Input 2 | Target |
|---:|---:|---:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

After training:

- probabilities near `0` represent the negative class;
- probabilities near `1` represent the positive class;
- the threshold `0.5` converts probabilities into class predictions;
- the final accuracy should be `100%` for this small training set.

## Why a Hidden Layer Is Required

XOR is not linearly separable. A single linear decision boundary cannot separate its positive and negative examples. The hidden layer learns an intermediate nonlinear representation that makes the final classification possible.

## Implementation Notes

This notebook is an instructional implementation rather than a production neural-network framework.

It intentionally omits:

- mini-batch training;
- explicit hidden and output bias vectors;
- alternative activation functions;
- advanced optimizers;
- validation data;
- regularization;
- numerical gradient checking;
- model serialization.

Libraries such as Keras and PyTorch provide these features, automatic differentiation, and optimized tensor operations.

## Exercises

1. Change the hidden-layer size and compare convergence.
2. Test different learning rates.
3. Add explicit bias vectors to both layers.
4. Plot the loss history.
5. Replace sigmoid with another activation in the hidden layer.
6. Add a method for saving and loading weights.
7. Compare this implementation with an equivalent Keras model.